# SMD Component Detector — Retrain & Export NMS Model

Downloads dataset → Trains YOLOv5n → Exports TFLite with built-in NMS.
**Run all cells — no manual steps.**

In [ ]:
!pip install -q roboflow ultralytics onnx onnxruntime onnx2tf
print("Dependencies ready")

In [ ]:
from roboflow import Roboflow
import os, yaml

rf = Roboflow(api_key="aLtYgQ8E7uKE7s3FNO20")
project = rf.workspace("dainius").project("smdcomponents")
dataset = project.version(6).download("yolov5")
print(f"Dataset: {dataset.location}")

# Fix data.yaml paths
for root, dirs, files in os.walk(dataset.location):
    if "data.yaml" in files:
        yaml_path = os.path.join(root, "data.yaml")
        break

with open(yaml_path) as f:
    cfg = yaml.safe_load(f)
base = os.path.dirname(yaml_path)
cfg['path'] = base
cfg['train'] = os.path.join(base, 'train')
cfg['val'] = os.path.join(base, 'valid')
if os.path.exists(os.path.join(base, 'test')):
    cfg['test'] = os.path.join(base, 'test')
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f)
print(f"data.yaml fixed at: {yaml_path}")

In [ ]:
!yolo detect train data={yaml_path} model=yolov5nu.pt epochs=20 imgsz=416 batch=32 name=smd_nms exist_ok=True device=0
print("Training done!")

In [ ]:
from ultralytics import YOLO
from google.colab import files
import glob

pt = glob.glob("runs/detect/smd_nms/weights/best.pt")[0]
print(f"Model: {pt}")

# Export with built-in NMS — output: [N, 6] = [x1, y1, x2, y2, class, score]
model = YOLO(pt)
model.export(format="tflite", nms=True, imgsz=416)

tfl = glob.glob("runs/detect/smd_nms/weights/*_nms.tflite") or glob.glob("runs/detect/smd_nms/weights/*.tflite")
if tfl:
    print(f"Downloading: {tfl[0]}")
    files.download(tfl[0])
else:
    print("No TFLite found — check export output")
    files.download(pt)